# 🛡️ VAJRA: 1-Click 7B Model Compression & GGUF Quantization
**Merges `AravKataria/vajra-lora` with `Qwen2.5-Coder-7B-Instruct` and quantizes to 4-bit `Q4_K_M` (~4.2 GB).**

### 📌 Instructions:
1. Ensure you are on a **GPU runtime**: Click **Runtime** -> **Change runtime type** -> select **T4 GPU** (Free).
2. Click **Runtime** -> **Run all** (or run each cell sequentially).
3. When prompted in Step 2, enter your Hugging Face Token (with **WRITE** permissions).
4. In ~6-8 minutes, your compressed model will be uploaded to `https://huggingface.co/AravKataria/vajra-7b-gguf`.

In [ ]:
# Step 1: Install required dependencies & fix Colab torchao conflict
!pip install -q -U torchao transformers peft torch accelerate huggingface_hub gguf sentencepiece protobuf
print('✅ Dependencies updated successfully!')

In [ ]:
# Step 2: Hugging Face Authentication
import os
from huggingface_hub import HfApi, login
from getpass import getpass

HF_TOKEN = os.getenv('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = getpass('🔑 Enter your Hugging Face Access Token (with WRITE permissions): ').strip()

login(token=HF_TOKEN)
print('✅ Successfully authenticated with Hugging Face!')

In [ ]:
# Step 3: Load Base Model and Merge LoRA Adapter
import os
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
LORA_MODEL = 'AravKataria/vajra-lora'
MERGED_DIR = './vajra_merged_7b'
OFFLOAD_DIR = './offload'

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(OFFLOAD_DIR, exist_ok=True)

print(f'[*] Loading tokenizer from {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN, trust_remote_code=True)
tokenizer.save_pretrained(MERGED_DIR)

print(f'[*] Loading base model {BASE_MODEL} in float16...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map='auto',
    offload_folder=OFFLOAD_DIR,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

print(f'[*] Merging fine-tuned LoRA weights from {LORA_MODEL}...')
model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL,
    token=HF_TOKEN,
    offload_folder=OFFLOAD_DIR
)
merged = model.merge_and_unload(safe_merge=True)

print(f'[*] Saving merged weights to {MERGED_DIR}...')
merged.save_pretrained(MERGED_DIR)
print('✅ LoRA weights successfully merged into standalone checkpoint!')

# Free memory before llama.cpp build and quantization
del base_model, model, merged
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Step 4: Clone & Build llama.cpp
import os
print('⚙️ Building llama.cpp tools with CMake...')
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j$(nproc)
!pip install -q -r llama.cpp/requirements.txt
print('✅ llama.cpp built successfully!')

In [ ]:
# Step 5: Convert to GGUF and Quantize to Q4_K_M (~4.2 GB)
print('📦 Converting merged checkpoint to intermediate GGUF (FP16)...')
!python llama.cpp/convert_hf_to_gguf.py ./vajra_merged_7b --outtype f16 --outfile vajra-7b-f16.gguf

print('⚡ Quantizing to 4-bit Q4_K_M (optimal balance of accuracy and memory)...')
!./llama.cpp/build/bin/llama-quantize vajra-7b-f16.gguf vajra-7b-q4_k_m.gguf Q4_K_M

# Clean up intermediate files to preserve disk space
!rm -f vajra-7b-f16.gguf
!rm -rf ./offload
print('✅ Quantization complete!')
!ls -lh vajra-7b-q4_k_m.gguf

In [ ]:
# Step 6: Upload Quantized GGUF Model to Hugging Face
DEST_REPO = 'AravKataria/vajra-7b-gguf'
print(f'🚀 Uploading vajra-7b-q4_k_m.gguf to Hugging Face ({DEST_REPO})...')

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=DEST_REPO, repo_type='model', exist_ok=True)

api.upload_file(
    path_or_fileobj='vajra-7b-q4_k_m.gguf',
    path_in_repo='vajra-7b-q4_k_m.gguf',
    repo_id=DEST_REPO,
    repo_type='model',
    commit_message='Add 4-bit Q4_K_M quantized GGUF weights (~4.2 GB)'
)

print('\n' + '='*65)
print('🎉 SUCCESS! Compressed VAJRA 7B GGUF Model is live:')
print(f'👉 https://huggingface.co/{DEST_REPO}')
print('='*65)